# Tournament Manager — Repository Tests

Tests every repository method using an **in-memory SQLite** database.
No external services required. Run all cells top to bottom.

Following the BandHub pattern: engine and Session are imported from `src.domain`.

## 0. Setup

In [ ]:
import os, sys

# Project root on sys.path (BandHub style: sys.path.append('../src'))
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Switch to test environment BEFORE importing anything from src.domain
os.environ['APP_ENV'] = 'test'
os.environ['DATABASE_URL_TEST'] = 'sqlite:///:memory:'

# Import engine, Session and Base from src.domain — same objects used everywhere
from src.domain.db import engine, Session, Base
from src.domain.models import (
    Fighter, MedicalRecord, Tournament, Tatami,
    Category, FighterCategoryRegistration, Match
)
from src.domain.repositories import UnitOfWork

# Create all tables in the in-memory database
Base.metadata.create_all(engine)

def make_uow():
    """Return a UnitOfWork wired to the test session."""
    return UnitOfWork(session_factory=Session)

print('✅ Setup complete — using:', engine)

---
## 1. FighterRepository
### 1.1 Create (add)

In [ ]:
with make_uow() as uow:
    f1 = Fighter(first_name='Ana', last_name='García', gender='F',
                 belt_level='blue', club_name='Club Judo BCN',
                 country='Spain', email='ana.garcia@example.com')
    f2 = Fighter(first_name='Carlos', last_name='López', gender='M',
                 belt_level='black', club_name='Dojo Madrid',
                 country='Spain', email='carlos.lopez@example.com')
    f3 = Fighter(first_name='Yuki', last_name='Tanaka', gender='F',
                 belt_level='purple', club_name='Osaka Dojo',
                 country='Japan', email='yuki.tanaka@example.com')
    uow.fighters.add(f1)
    uow.fighters.add(f2)
    uow.fighters.add(f3)
    uow.commit()
    fighter_ids = [f1.fighter_id, f2.fighter_id, f3.fighter_id]

print('✅ Created fighters:', fighter_ids)

### 1.2 Read by ID

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    assert fighter is not None
    assert fighter.first_name == 'Ana'
    print(f'✅ get() → {fighter}')

### 1.3 Read all

In [ ]:
with make_uow() as uow:
    all_fighters = uow.fighters.get_all()
    assert len(all_fighters) == 3
    print(f'✅ get_all() → {len(all_fighters)} fighters')

### 1.4 Update

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    fighter.club_name = 'Club Judo Barcelona Updated'
    uow.fighters.update(fighter)
    uow.commit()

with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    assert fighter.club_name == 'Club Judo Barcelona Updated'
    assert fighter.last_update is not None, 'last_update should be set after update()'
    print(f'✅ update() → club_name updated, last_update={fighter.last_update}')

### 1.5 Specific queries

In [ ]:
with make_uow() as uow:
    results = uow.fighters.get_by_last_name('García')
    assert len(results) == 1
    print(f'✅ get_by_last_name → {results[0]}')

    results = uow.fighters.get_by_belt_level('black')
    assert len(results) == 1
    print(f'✅ get_by_belt_level → {results[0]}')

    results = uow.fighters.get_by_country('Spain')
    assert len(results) == 2
    print(f'✅ get_by_country → {len(results)} fighters')

    fighter = uow.fighters.get_by_email('yuki.tanaka@example.com')
    assert fighter is not None
    print(f'✅ get_by_email → {fighter}')

### 1.6 Pagination

In [ ]:
with make_uow() as uow:
    page1 = uow.fighters.get_paginated(page=1, page_size=2)
    page2 = uow.fighters.get_paginated(page=2, page_size=2)
    total  = uow.fighters.count()

    assert len(page1) == 2
    assert len(page2) == 1
    assert total == 3
    print(f'✅ Pagination: page1={[f.last_name for f in page1]}, page2={[f.last_name for f in page2]}')
    print(f'✅ count() → {total}')

### 1.7 Domain op — add_medical_record

In [ ]:
with make_uow() as uow:
    record = uow.fighters.add_medical_record(
        fighter_id=fighter_ids[0],
        blood_type='A+', notes='No known conditions', doctor_name='Dr. Martínez')
    uow.commit()
    print(f'✅ add_medical_record() → {record}')

with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[0])
    assert fighter.medical_record is not None
    assert fighter.medical_record.blood_type == 'A+'
    print(f'✅ Medical record attached: blood_type={fighter.medical_record.blood_type}')

### 1.8 Delete

In [ ]:
with make_uow() as uow:
    fighter = uow.fighters.get(fighter_ids[2])
    uow.fighters.delete(fighter)
    uow.commit()

with make_uow() as uow:
    assert uow.fighters.get(fighter_ids[2]) is None
    assert uow.fighters.count() == 2
    print('✅ delete() → fighter removed')

---
## 2. TournamentRepository

In [ ]:
from datetime import datetime, timedelta, timezone
now = datetime.now(timezone.utc).replace(tzinfo=None)

with make_uow() as uow:
    t1 = Tournament(name='Copa Primavera 2026', location='Barcelona',
                    start_date=now + timedelta(days=30),
                    end_date=now + timedelta(days=31), max_fighters=64)
    t2 = Tournament(name='Grand Prix Madrid', location='Madrid',
                    start_date=now + timedelta(days=60),
                    end_date=now + timedelta(days=61), max_fighters=128)
    uow.tournaments.add(t1)
    uow.tournaments.add(t2)
    uow.commit()
    tournament_ids = [t1.tournament_id, t2.tournament_id]
    print(f'✅ Created: {t1}, {t2}')

with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[0])
    assert t.name == 'Copa Primavera 2026'
    assert len(uow.tournaments.get_all()) == 2
    print(f'✅ get() and get_all() OK')

with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[0])
    t.max_fighters = 80
    uow.tournaments.update(t)
    uow.commit()

with make_uow() as uow:
    assert uow.tournaments.get(tournament_ids[0]).max_fighters == 80
    print('✅ update() OK')
    assert len(uow.tournaments.get_by_name('Copa')) == 1
    assert len(uow.tournaments.get_by_location('Madrid')) == 1
    assert len(uow.tournaments.get_active_tournaments()) >= 2
    print('✅ get_by_name / get_by_location / get_active_tournaments OK')

with make_uow() as uow:
    tatami = uow.tournaments.add_tatami_to_tournament(
        tournament_ids[0], name='Tatami A', mat_number=1, area_size=64.0)
    uow.commit()
    print(f'✅ add_tatami_to_tournament() → {tatami}')

with make_uow() as uow:
    t = uow.tournaments.get(tournament_ids[1])
    uow.tournaments.delete(t)
    uow.commit()
    assert uow.tournaments.get(tournament_ids[1]) is None
    print('✅ delete() OK')

---
## 3. CategoryRepository

In [ ]:
with make_uow() as uow:
    c1 = Category(name='Blue -57kg Male',    belt_level='blue',  weight_min_kg=0,  weight_max_kg=57,  gender='M')
    c2 = Category(name='Black -70kg Female', belt_level='black', weight_min_kg=57, weight_max_kg=70,  gender='F')
    c3 = Category(name='Blue -70kg Female',  belt_level='blue',  weight_min_kg=57, weight_max_kg=70,  gender='F')
    uow.categories.add(c1)
    uow.categories.add(c2)
    uow.categories.add(c3)
    uow.commit()
    cat_ids = [c1.category_id, c2.category_id, c3.category_id]

with make_uow() as uow:
    assert uow.categories.get(cat_ids[0]) is not None
    assert len(uow.categories.get_all()) == 3
    assert len(uow.categories.get_by_belt_level('blue')) == 2
    assert len(uow.categories.get_by_gender('F')) == 2
    assert len(uow.categories.get_by_name('Black')) == 1
    by_weight = uow.categories.get_by_weight_range(max_kg=60)
    assert len(by_weight) == 1
    print('✅ CategoryRepository: all queries OK')

---
## 4. RegistrationRepository

In [ ]:
# Re-read IDs (Yuki was deleted, only fighter_ids[0] and [1] remain)
with make_uow() as uow:
    all_f = uow.fighters.get_all()
    fid0, fid1 = all_f[0].fighter_id, all_f[1].fighter_id
    tid = tournament_ids[0]
    cid0, cid1 = cat_ids[0], cat_ids[2]

    reg1 = FighterCategoryRegistration(
        fighter_id=fid0, category_id=cid1, tournament_id=tid,
        weight_in_weight_kg=56.2, is_approved=False)
    reg2 = FighterCategoryRegistration(
        fighter_id=fid1, category_id=cid0, tournament_id=tid,
        weight_in_weight_kg=55.0, is_approved=False)
    uow.registrations.add(reg1)
    uow.registrations.add(reg2)
    uow.commit()

with make_uow() as uow:
    reg = uow.registrations.get((fid0, cid1, tid))
    assert reg is not None
    assert len(uow.registrations.get_by_tournament(tid)) == 2
    assert len(uow.registrations.get_pending_approvals(tid)) == 2
    print('✅ RegistrationRepository: get, get_by_tournament, get_pending_approvals OK')

with make_uow() as uow:
    uow.fighters.approve_registration(fid0, cid1, tid)
    uow.commit()

with make_uow() as uow:
    assert len(uow.registrations.get_pending_approvals(tid)) == 1
    print('✅ approve_registration() domain op OK')

---
## 5. MatchRepository

In [ ]:
with make_uow() as uow:
    m1 = Match(tournament_id=tournament_ids[0], category_id=cat_ids[2],
               blue_fighter_id=fid0, red_fighter_id=fid1,
               winner_id=fid0, round_number=1,
               duration_seconds=185.0, win_method='ippon',
               scheduled_time=datetime.now(timezone.utc).replace(tzinfo=None) + timedelta(days=30, hours=9))
    m2 = Match(tournament_id=tournament_ids[0], category_id=cat_ids[0],
               blue_fighter_id=fid1, red_fighter_id=fid0,
               winner_id=fid1, round_number=2,
               duration_seconds=245.0, win_method='waza-ari',
               scheduled_time=datetime.now(timezone.utc).replace(tzinfo=None) + timedelta(days=30, hours=10))
    uow.matches.add(m1)
    uow.matches.add(m2)
    uow.commit()
    match_ids = [m1.match_id, m2.match_id]

with make_uow() as uow:
    assert uow.matches.get(match_ids[0]) is not None
    assert len(uow.matches.get_all()) == 2
    assert len(uow.matches.get_by_tournament(tournament_ids[0])) == 2
    assert len(uow.matches.get_by_fighter(fid0)) == 2
    assert len(uow.matches.get_by_category(cat_ids[2])) == 1
    assert len(uow.matches.get_by_round(tournament_ids[0], 1)) == 1
    assert len(uow.matches.get_wins_by_fighter(fid0)) == 1
    print('✅ MatchRepository: all queries OK')

---
## 6. Rollback test

In [ ]:
with make_uow() as uow:
    count_before = uow.fighters.count()

try:
    with make_uow() as uow:
        uow.fighters.add(Fighter(first_name='Ghost', last_name='Fighter'))
        raise RuntimeError('Simulated error — should rollback')
except RuntimeError:
    pass

with make_uow() as uow:
    count_after = uow.fighters.count()

assert count_after == count_before
print(f'✅ Rollback works — count unchanged at {count_after}')

In [ ]:
print('=' * 50)
print('ALL REPOSITORY TESTS PASSED ✅')
print('=' * 50)